In [44]:
import torch
import numpy as np


# x = torch.rand((2, 50000, 2000)).to('cuda').float()
# y = torch.rand((2, 20000, 2000)).to('cuda').float()

# W = torch.bmm(x, y.transpose(2, 1))
# yy = torch.bmm(W, y)


# x = torch.tensor([[[[1, 2], [2, 3]],[[4, 5], [5, 6]]], [[[1, 2], [2, 3]],[[4, 5], [5, 6]]]])
# x = torch.rand((2, 256, 120, 160)).to('cuda').float()
# xr = x.view((x.shape[0], x.shape[1], -1))

x = torch.rand((8, 10000, 30)).to('cuda').float()
y = torch.rand((8, 30, 12000)).to('cuda').float()
z = torch.rand((8, 12000, 40)).to('cuda').float()

def batch_matMul(x, y, z):
    w = torch.bmm(x, y)
    zw1 = torch.bmm(w, z)
    return zw1



def einsum_batch_matMul(x, y, z):
    zw2 = torch.einsum('bij,bjk,bkl->bil', [x, y, z])
    return zw2


def split_batch_matMul(x, y, z):
    n_pts = x.size()[1]
    n_limit = 5000
    n = n_pts/n_limit
    x_list = torch.split(x, int(np.ceil(n_pts/n)), dim=1) 
    zw = torch.tensor([]).to('cuda').float()
    for i in range(int(n)):
        x_part = x_list[i]
        print(x_part.size())
        w = torch.bmm(x_part, y)
        zw3 = torch.bmm(w, z)
        zw = torch.cat((zw, zw3), dim=1)
    return zw
# xm = torch.max(x, 1, keepdim=True)[0] 
# xr = xm.view(-1, 18)

# x = torch.tensor([[10, 0], [100, 0]]).float()



In [49]:
%time batch_matMul(x, y, z)

CPU times: user 1.2 ms, sys: 16.6 ms, total: 17.8 ms
Wall time: 16.5 ms


tensor([[[42433.1680, 42932.6680, 42822.4141,  ..., 42473.9766,
          42582.0781, 43005.1055],
         [43453.7305, 43926.8750, 43825.6055,  ..., 43480.5586,
          43569.1680, 44066.9414],
         [43362.5703, 43882.9609, 43761.5664,  ..., 43426.0078,
          43468.9570, 43976.7773],
         ...,
         [40471.7227, 40937.1523, 40834.7852,  ..., 40554.8906,
          40616.1836, 41051.8672],
         [52439.8516, 53055.1250, 52893.1055,  ..., 52508.5938,
          52631.5391, 53160.2930],
         [47847.8555, 48382.9102, 48281.1328,  ..., 47924.4336,
          47977.5547, 48493.7539]],

        [[49146.7266, 50074.5586, 49061.9844,  ..., 49052.4922,
          48855.0469, 49487.3594],
         [43715.2227, 44541.5625, 43625.1055,  ..., 43644.9219,
          43474.0469, 44037.9688],
         [44821.2852, 45640.7969, 44724.8203,  ..., 44737.6992,
          44581.5820, 45139.8086],
         ...,
         [43507.1367, 44347.0117, 43438.4570,  ..., 43451.9648,
          43274

In [50]:
%time einsum_batch_matMul(x, y, z)

CPU times: user 855 µs, sys: 237 µs, total: 1.09 ms
Wall time: 691 µs


tensor([[[42433.1680, 42932.6680, 42822.4141,  ..., 42473.9766,
          42582.0781, 43005.1055],
         [43453.7305, 43926.8750, 43825.6055,  ..., 43480.5586,
          43569.1680, 44066.9414],
         [43362.5703, 43882.9609, 43761.5664,  ..., 43426.0078,
          43468.9570, 43976.7773],
         ...,
         [40471.7227, 40937.1523, 40834.7852,  ..., 40554.8906,
          40616.1836, 41051.8672],
         [52439.8516, 53055.1250, 52893.1055,  ..., 52508.5938,
          52631.5391, 53160.2930],
         [47847.8555, 48382.9102, 48281.1328,  ..., 47924.4336,
          47977.5547, 48493.7539]],

        [[49146.7266, 50074.5586, 49061.9844,  ..., 49052.4922,
          48855.0469, 49487.3594],
         [43715.2227, 44541.5625, 43625.1055,  ..., 43644.9219,
          43474.0469, 44037.9688],
         [44821.2852, 45640.7969, 44724.8203,  ..., 44737.6992,
          44581.5820, 45139.8086],
         ...,
         [43507.1367, 44347.0117, 43438.4570,  ..., 43451.9648,
          43274

In [48]:
%time split_batch_matMul(x, y, z)

torch.Size([8, 5000, 30])
torch.Size([8, 5000, 30])
CPU times: user 1.01 ms, sys: 277 µs, total: 1.29 ms
Wall time: 810 µs


tensor([[[42433.1680, 42932.6680, 42822.4141,  ..., 42473.9766,
          42582.0781, 43005.1055],
         [43453.7305, 43926.8750, 43825.6055,  ..., 43480.5586,
          43569.1680, 44066.9414],
         [43362.5703, 43882.9609, 43761.5664,  ..., 43426.0078,
          43468.9570, 43976.7773],
         ...,
         [40471.7227, 40937.1523, 40834.7852,  ..., 40554.8906,
          40616.1836, 41051.8672],
         [52439.8516, 53055.1250, 52893.1055,  ..., 52508.5938,
          52631.5391, 53160.2930],
         [47847.8555, 48382.9102, 48281.1328,  ..., 47924.4336,
          47977.5547, 48493.7539]],

        [[49146.7266, 50074.5586, 49061.9844,  ..., 49052.4922,
          48855.0469, 49487.3594],
         [43715.2227, 44541.5625, 43625.1055,  ..., 43644.9219,
          43474.0469, 44037.9688],
         [44821.2852, 45640.7969, 44724.8203,  ..., 44737.6992,
          44581.5820, 45139.8086],
         ...,
         [43507.1367, 44347.0117, 43438.4570,  ..., 43451.9648,
          43274

In [7]:
import torch
import numpy as np


# x = torch.rand((2, 50000, 2000)).to('cuda').float()
# y = torch.rand((2, 20000, 2000)).to('cuda').float()

# W = torch.bmm(x, y.transpose(2, 1))
# yy = torch.bmm(W, y)


# x = torch.tensor([[[[1, 2], [2, 3]],[[4, 5], [5, 6]]], [[[1, 2], [2, 3]],[[4, 5], [5, 6]]]])
# x = torch.rand((2, 256, 120, 160)).to('cuda').float()
# xr = x.view((x.shape[0], x.shape[1], -1))

x = torch.rand((2, 8, 3)).float()
y = torch.rand((2, 8, 6)).to('cuda').float()

Wq1_linear = torch.nn.Linear(3, 4)  

x_out = Wq1_linear(x)
print(x_out.shape)

torch.Size([2, 8, 4])


In [2]:
x = [1,2,3,4,5]
x[-]

4